In [1]:
import pandas as pd
import itertools

In [2]:
total_data = pd.read_csv('/home/s6moakba/Instruct_Restaurants.csv')

In [3]:
total_data
# 2697, 2703
filtered_rows_toal = total_data[total_data['aspect_terms'].apply(lambda x: len(x[1:-1].split(',')) > 2)]
filtered_rows_toal.loc[2697]['raw_words']

"Each time we 've been , the front of house staff ( not the waiters - they 're fantastic - but the people who greet and seat you ) has been so hideous to us that were it not for the exceptional fish dishes I would never return ."

In [4]:
test_prompt = """You are a critic who can generate comments on the specified aspect and sentiment
We would like you to complete a sentence generation task. Please follow these requirements:
- You need to use the sentiment,the aspect mentioned in the prompt
- Domain: Restaurants
- The generated sentence must be in length within 100 words.
- the sentence should not have aspect words that are not specified in the prompt 
- DO NOT REPEAT the input text in the output
- PRINT ONLY THE ANSWER TEXT NO EXPLAINING NOTHING ELSE, MAKE SURE TO USE ASPECT WORDS IN THE OUTPUT
- examples:
### input ###
aspect: prices
polarity: negative
### Output ### 
The prices were too high for this type of restaurant
### input ###
aspect: vibe, owner, service
polarity: positive,positive,negative
### Output ### 
"Best of all is the warm vibe , the owner is super friendly but service isn't fast ."
### input ###
aspect: bar, table, dinner
polarity: Positive, Neutral, Neutral
### Output ### 
After really enjoying ourselves at the bar we sat down at a table and had dinner

Now complete this task like example with ONE SENTENCE:
### Input ###"""

In [5]:

def preprocess_data(df):
    data = []
    for _, row in df.iterrows():
        combined_aspect_terms = row['aspect_terms'][1:-1].replace("'", "").split(", ")
        polarity_options = ['Positive', 'Neutral', 'Negative']
        polarity_combinations = list(itertools.product(polarity_options, repeat=len(combined_aspect_terms)))
        for polarity_combination in polarity_combinations:
            combined_aspects = ','.join(combined_aspect_terms)
            combined_polarities = ','.join(polarity_combination)
            prompt = (
                f"You are a critic who can generate comments on the specified aspect and sentiment\n"
                f"We would like you to complete a sentence generation task. Please follow these requirements:\n"
                f"- You need to use the sentiment,the aspect mentioned in the prompt\n"
                f"- Domain: Restaurants\n"
                f"- The generated sentence must be in length within 100 words.\n"
                f"- the sentence should not have aspect words that are not specified in the prompt\n"
                f"- DO NOT REPEAT the input text in the output\n"
                f"- PRINT ONLY THE ANSWER TEXT NO EXPLAINING NOTHING ELSE, MAKE SURE TO USE ASPECT WORDS IN THE OUTPUT\n"
                f"- examples:\n"
                f"### input ###\n"
                f"aspect: prices\n"
                f"polarity: negative\n"
                f"### Output ###\n"
                f"The prices were too high for this type of restaurant\n"
                f"### input ###\n"
                f"aspect: vibe, owner, service\n"
                f"polarity: positive,positive,negative\n"
                f"### Output ###\n"
                f"Best of all is the warm vibe , the owner is super friendly but service isn't fast .\n"
                f"### input ###\n"
                f"aspect: bar, table, dinner\n"
                f"polarity: Positive, Neutral, Neutral\n"
                f"### Output ###\n"
                f"After really enjoying ourselves at the bar we sat down at a table and had dinner\n"
                f"Now complete this task like example with ONE SENTENCE:\n"
                f"### Input ###\naspect: {combined_aspects}\npolarity: {combined_polarities}\n###Output### "
            )
            data.append({
                'aspect': combined_aspects,
                'polarity': combined_polarities,
                'original_text': row['raw_words'],
                'prompt': prompt
            })
    return pd.DataFrame(data)

# Preprocess data before feeding to model
processed_df = preprocess_data(total_data)

In [6]:
processed_df.shape

(55380, 4)

In [7]:
filtered_rows_1_aspect = processed_df[processed_df['aspect'].apply(lambda x: len(x.split(',')) == 1)]
filtered_rows_2_aspect = processed_df[processed_df['aspect'].apply(lambda x: len(x.split(',')) == 2)]
filtered_rows_3_aspect = processed_df[processed_df['aspect'].apply(lambda x: len(x.split(',')) == 3)]
filtered_rows_4_aspect = processed_df[processed_df['aspect'].apply(lambda x: len(x.split(',')) > 3)]

In [8]:
filtered_rows_1_aspect.shape

(4332, 4)

In [47]:
sample_1 = filtered_rows_1_aspect.sample(n=5000,replace=True)
sample_2 = filtered_rows_2_aspect.sample(n=10000,replace=True)
sample_3 = filtered_rows_3_aspect.sample(n=15000,replace=True)
sample_4 = filtered_rows_4_aspect.sample(n=5000,replace=True)

In [48]:
all_samples = pd.concat([sample_1, sample_2, sample_3])

In [49]:
all_samples

,aspect,polarity,original_text,prompt
37807,burgers,Neutral,"Consequently , their burgers fell apart in the...",You are a critic who can generate comments on ...
23398,crabmeat,Neutral,My entree of hot pot with seafood was full of ...,You are a critic who can generate comments on ...
51004,atomosphere,Neutral,Cozy romantic atomosphere with only around 15 ...,You are a critic who can generate comments on ...
49739,smoked salmon and roe appetizer,Negative,I ordered the smoked salmon and roe appetizer ...,You are a critic who can generate comments on ...
45979,place,Neutral,"First of all , this place is *not* romantic , ...",You are a critic who can generate comments on ...
...,...,...,...,...
48424,"back patio,back patio,music","Neutral,Positive,Neutral","We ate out in the back patio , which is worth ...",You are a critic who can generate comments on ...
26384,"service,food,atmosphere","Neutral,Negative,Negative","You do n't go to Mizu for excellent service , ...",You are a critic who can generate comments on ...
20933,"scallops,appetizer,sauce","Positive,Neutral,Negative",We had the scallops as an appetizer and they w...,You are a critic who can generate comments on ...
52731,"waiter,food,views of the city","Negative,Neutral,Positive","The waiter was attentive , the food was delici...",You are a critic who can generate comments on ...


In [51]:
all_samples.to_csv('/home/s6moakba/Instruct_Restaurants_samples.csv')

In [2]:
generated_sample = pd.read_csv('/home/s6moakba/generated_data_sample.csv')

NameError: name 'pd' is not defined

In [1]:
row = generated_sample.iloc[10]
print(row['aspect'])
print(row['polarity'])
print(row['generated_text'])

NameError: name 'generated_sample' is not defined

# Check results


In [9]:
generated_sample = pd.read_csv("/home/s6moakba/generated_data_eficient_dist_improve.csv")

In [10]:
generated_sample.shape

(30000, 5)

In [11]:
row = generated_sample.iloc[-2000]
print(row['aspect'])
print(row['polarity'])
print(row['generated_text'])
print(row['original_text'])

meal,meal,restaurant
Neutral,Positive,Negative
The meal was just okay, but the restaurant's specialty dish was surprisingly delicious, and unfortunately, the dessert was a letdown.
The restaurant is a bit noisy but that is something that can be overlooked once you sit down and enjoy a great meal
